# Кейс Alpha-Bank - маскирование персональный данных

## Шаг 1 — Загружаем kaggle.json


In [2]:
from google.colab import files
import os

print('Загрузи файл kaggle.json')
uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✓ kaggle.json установлен')

Загрузи файл kaggle.json


Saving kaggle.json to kaggle.json
✓ kaggle.json установлен


## Шаг 2 — Скачиваем датасеты

In [3]:
!kaggle competitions download -c llm-march2026-alfabank -p /content/data

100% 587k/587k [00:00<00:00, 608kB/s]



In [4]:
import zipfile, os

zip_path = "/content/data/llm-march2026-alfabank.zip"
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/data")

for f in os.listdir("/content/data"):
    print(f)

private_test_dataset.csv
train_dataset.tsv
llm-march2026-alfabank.zip


In [5]:
import pandas as pd
import ast

train_df = pd.read_csv("/content/data/train_dataset.tsv", sep="\t")

print("Размер:", train_df.shape)
print("Колонки:", train_df.columns.tolist())
print("Первые 5 колонок:")
train_df.head(5)

Размер: (8287, 3)
Колонки: ['text', 'target', 'entity']
Первые 5 колонок:


,text,target,entity
0,Вы можете обновить client secret в настройках ...,[],empty
1,"Возможно, произошел временный сбой на стороне ...",[],empty
2,Наши запросы с API ключом bk_api_key_ABCDEF123...,"[(26, 69, 'API ключи')]",['bk_api_key_ABCDEF1234567890ABCDEF1234567890']
3,Сгенерируйте новый JWT для сервиса аутентифика...,[],empty
4,"Возможно, токен 6363069502:y-3-y-6-z-0-w-5-w-0...","[(16, 100, 'API ключи')]",['6363069502:y-3-y-6-z-0-w-5-w-0-x-1-z-1-x-1-x...


## Шаг 3 - BIO схема для меток

### Подсчет уникальных тегов

In [6]:
from collections import Counter
cat_counts = Counter()

for val in train_df["target"]:
    if pd.isna(val) or val == "[]":
        continue
    spans = ast.literal_eval(val)
    for span in spans:
        cat_counts[span[2]] += 1

print(f"Всего уникальных категорий: {len(cat_counts)}\n")
for cat, cnt in cat_counts.most_common():
    print(f"  {cnt:4d}  {cat}")

Всего уникальных категорий: 30

   811  Полный адрес
   362  Дата рождения
   337  API ключи
   333  Гражданство и названия стран
   324  Место рождения
   307  Разрешение на работу / визу
   302  Данные об автомобиле клиента
   296  Данные об организации/юридическом лице (ИНН, КПП, ОГРН, БИК, адреса, расчётный счёт)
   280  Номер телефона
   278  Водительское удостоверение
   270  Сведения об ИНН
   256  Паспортные данные
   254  ФИО
   238  Дата окончания срока действия карты
   225  Номер банковского счета
   208  CVV/CVC
   192  Дата регистрации по месту жительства или пребывания
   172  Содержимое магнитной полосы
   171  Имя держателя карты
   165  Временное удостоверение личности
   163  Наименование банка
   146  Пароли
   143  Кодовые слова
   141  Номер карты
   141  ПИН код
   139  Одноразовые коды
   138  Свидетельство о рождении
   133  СНИЛС клиента
   129  Email
   104  Серия и номер вида на жительство


In [7]:
CATEGORIES = [
    "API ключи",
    "CVV/CVC",
    "Email",
    "Водительское удостоверение",
    "Временное удостоверение личности",
    "Гражданство и названия стран",
    "Данные об автомобиле клиента",
    "Данные об организации/юридическом лице (ИНН, КПП, ОГРН, БИК, адреса, расчётный счёт)",
    "Дата окончания срока действия карты",
    "Дата регистрации по месту жительства или пребывания",
    "Дата рождения",
    "Имя держателя карты",
    "Кодовые слова",
    "Место рождения",
    "Наименование банка",
    "Номер банковского счета",
    "Номер карты",
    "Номер телефона",
    "Одноразовые коды",
    "ПИН код",
    "Пароли",
    "Паспортные данные",
    "Полный адрес",
    "Разрешение на работу / визу",
    "СНИЛС клиента",
    "Сведения об ИНН",
    "Свидетельство о рождении",
    "Серия и номер вида на жительство",
    "Содержимое магнитной полосы",
    "ФИО",
]

### Нормализация меток

In [8]:
def normalize_label(cat: str) -> str:
    for ch in " /(),.":
        cat = cat.replace(ch, "_")
    while "__" in cat:
        cat = cat.replace("__", "_")
    return cat.strip("_")
cat2norm = {cat: normalize_label(cat) for cat in CATEGORIES}
norm2cat = {norm: cat for cat, norm in cat2norm.items()}
print(cat2norm)

{'API ключи': 'API_ключи', 'CVV/CVC': 'CVV_CVC', 'Email': 'Email', 'Водительское удостоверение': 'Водительское_удостоверение', 'Временное удостоверение личности': 'Временное_удостоверение_личности', 'Гражданство и названия стран': 'Гражданство_и_названия_стран', 'Данные об автомобиле клиента': 'Данные_об_автомобиле_клиента', 'Данные об организации/юридическом лице (ИНН, КПП, ОГРН, БИК, адреса, расчётный счёт)': 'Данные_об_организации_юридическом_лице_ИНН_КПП_ОГРН_БИК_адреса_расчётный_счёт', 'Дата окончания срока действия карты': 'Дата_окончания_срока_действия_карты', 'Дата регистрации по месту жительства или пребывания': 'Дата_регистрации_по_месту_жительства_или_пребывания', 'Дата рождения': 'Дата_рождения', 'Имя держателя карты': 'Имя_держателя_карты', 'Кодовые слова': 'Кодовые_слова', 'Место рождения': 'Место_рождения', 'Наименование банка': 'Наименование_банка', 'Номер банковского счета': 'Номер_банковского_счета', 'Номер карты': 'Номер_карты', 'Номер телефона': 'Номер_телефона', 'О

### BIO-cхема  
Каждому токену присваивается один из трёх префиксов:
- **B** (Beginning) — первый токен сущности
- **I** (Inside) — продолжение сущности
- **O** (Outside) — не сущность

Пример для фразы "Токен 9876543210:AAE был создан`:

| Токен      | Метка              |
|------------|--------------------|
| Токен      | O                  |
| 9876543210 | B-API_ключи        |
| :          | I-API_ключи        |
| AAE        | I-API_ключи        |
| был        | O                  |
| создан     | O                  |

In [9]:
labels = ["O"] + [f"{bio}-{normalize_label(cat)}"
                  for cat in CATEGORIES
                  for bio in ("B", "I")]

label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}

print(f"Всего меток: {len(labels)}")
print(f"Метки: {labels}")

Всего меток: 61
Метки: ['O', 'B-API_ключи', 'I-API_ключи', 'B-CVV_CVC', 'I-CVV_CVC', 'B-Email', 'I-Email', 'B-Водительское_удостоверение', 'I-Водительское_удостоверение', 'B-Временное_удостоверение_личности', 'I-Временное_удостоверение_личности', 'B-Гражданство_и_названия_стран', 'I-Гражданство_и_названия_стран', 'B-Данные_об_автомобиле_клиента', 'I-Данные_об_автомобиле_клиента', 'B-Данные_об_организации_юридическом_лице_ИНН_КПП_ОГРН_БИК_адреса_расчётный_счёт', 'I-Данные_об_организации_юридическом_лице_ИНН_КПП_ОГРН_БИК_адреса_расчётный_счёт', 'B-Дата_окончания_срока_действия_карты', 'I-Дата_окончания_срока_действия_карты', 'B-Дата_регистрации_по_месту_жительства_или_пребывания', 'I-Дата_регистрации_по_месту_жительства_или_пребывания', 'B-Дата_рождения', 'I-Дата_рождения', 'B-Имя_держателя_карты', 'I-Имя_держателя_карты', 'B-Кодовые_слова', 'I-Кодовые_слова', 'B-Место_рождения', 'I-Место_рождения', 'B-Наименование_банка', 'I-Наименование_банка', 'B-Номер_банковского_счета', 'I-Номер_бан

## Шаг 4 - Токенизация

In [10]:
%pip install transformers datasets seqeval accelerate torch --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Токенизатор AutoTokenizer для BERT-моделей работает на основе алгоритма **WordPiece**. На каждом шаге объединяется та пара символов/подслов, которая
встречается непропорционально часто относительно каждого элемента по отдельности:  
`score(A, B) = count(AB) / (count(A) · count(B))`  
Пара с максимальным score добавляется в словарь, процесс повторяется до достижения нужного размера словаря

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased-conversational")
print(f"Токенизатор загружен: {"DeepPavlov/rubert-base-cased-conversational"}")
print(f"Словарь: {tokenizer.vocab_size:,} токенов")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Токенизатор загружен: DeepPavlov/rubert-base-cased-conversational
Словарь: 100,792 токенов


Функция spans_to_bio принимает текст и символьные спаны, токенизирует текст и через offset_mapping: назначает каждому токену BIO-метку: `B-` первому токену спана, `I-` остальным, `-100` спецтокенам.
Возвращает `input_ids`, `attention_mask` и `labels`

In [12]:
from typing import List, Tuple

MAX_LENGTH = 128

def spans_to_bio(text: str, spans: List[Tuple[int, int, str]]) -> dict:
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_offsets_mapping=True,
    )

    offset_mapping = encoding["offset_mapping"]
    input_ids      = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]

    label_ids = [label2id["O"]] * len(input_ids)

    # Для каждого спана находим индекс первого токена
    for span_start, span_end, category in spans:
        norm = cat2norm.get(category)
        if norm is None:
            continue

        first_token = True
        for token_idx, (tok_start, tok_end) in enumerate(offset_mapping):
            if tok_start == 0 and tok_end == 0:
                continue
            if tok_start >= span_start and tok_end <= span_end:
                prefix = "B" if first_token else "I"
                label_ids[token_idx] = label2id[f"{prefix}-{norm}"]
                first_token = False

    # Спецтокены и паддинг → -100
    for token_idx, (tok_start, tok_end) in enumerate(offset_mapping):
        if tok_start == 0 and tok_end == 0:
            label_ids[token_idx] = -100

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         label_ids,
    }

In [13]:
text  = "Клиент Иванов Иван обратился с номера +7 (903) 123-45-67"
spans = [(7, 19, "ФИО"), (37, 56, "Номер телефона")]

out = spans_to_bio(text, spans)
tokens     = tokenizer.convert_ids_to_tokens(out["input_ids"])
label_strs = [id2label.get(l, "IGN") for l in out["labels"]]

print(f"{'Токен':<22} {'Лейбл'}")
print("-" * 45)
for tok, lbl in zip(tokens, label_strs):
    if tok != "[PAD]":
        print(f"{tok:<22} {lbl}")

Токен                  Лейбл
---------------------------------------------
[CLS]                  IGN
Клиент                 O
Иванов                 B-ФИО
Иван                   I-ФИО
обратился              O
с                      O
номера                 O
+                      B-Номер_телефона
7                      I-Номер_телефона
(                      I-Номер_телефона
90                     I-Номер_телефона
##3                    I-Номер_телефона
)                      I-Номер_телефона
123                    I-Номер_телефона
-                      I-Номер_телефона
45                     I-Номер_телефона
-                      I-Номер_телефона
67                     I-Номер_телефона
[SEP]                  IGN


Применяем препроцессинг ко всему датасету

In [14]:
processed = []
for _, row in train_df.iterrows():
    spans = [] if pd.isna(row["target"]) or row["target"] == "[]" else ast.literal_eval(row["target"])
    processed.append(spans_to_bio(row["text"], spans))

print(f"Готово: {len(processed):,} примеров")

Готово: 8,287 примеров


## Шаг 5 - Формирование датасета HuggingFace

Разбивка train/val 90/10

In [15]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    processed,
    test_size=0.1,
    random_state=42,
)

def to_hf_dataset(records: list) -> Dataset:
    return Dataset.from_dict({
        "input_ids":      [r["input_ids"]      for r in records],
        "attention_mask": [r["attention_mask"] for r in records],
        "labels":         [r["labels"]         for r in records],
    })

dataset = DatasetDict({
    "train":      to_hf_dataset(train_data),
    "validation": to_hf_dataset(val_data),
})

print(f"Train:      {len(dataset['train']):,} примеров")
print(f"Validation: {len(dataset['validation']):,} примеров")

Train:      7,458 примеров
Validation: 829 примеров


## Шаг 6 - Обучение модели

In [16]:
!pip install seqeval -q

**Верный расчет F1-score для соревнования:**  
Oценка проводится по строгому совпадению границ и категории сущностей (strict span+category match). Основная метрика — micro-averaged F1-score.  

Сущность считается правильно найденной, только если:
- TP (True Positive): Предсказанная сущность полностью совпадает с эталонной по границам и категории
- FP (False Positive): Предсказанная сущность не совпадает ни с одной эталонной (ошибка в границах или категории, либо лишняя сущность)
- FN (False Negative): Эталонная сущность не найдена ни одним предсказанием.
TP, FP, FN считаются по всем категориям и строкам вместе

In [17]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels, pred_labels = [], []
    for pred_seq, label_seq in zip(predictions, label_ids):
        true_row, pred_row = [], []
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            true_row.append(id2label[l])
            pred_row.append(id2label[p])
        true_labels.append(true_row)
        pred_labels.append(pred_row)

    return {
        "f1":        f1_score(true_labels, pred_labels),
        "precision": precision_score(true_labels, pred_labels),
        "recall":    recall_score(true_labels, pred_labels),
    }

print("compute_metrics готова")

compute_metrics готова


In [18]:
import torch
from transformers import AutoModelForTokenClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {device}")

model = AutoModelForTokenClassification.from_pretrained(
    "DeepPavlov/rubert-base-cased-conversational",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

Устройство: cuda


pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-conversational
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias           

Параметров: 177,309,757


Настройка обучения

**Оптимизатор:** AdamW (по умолчанию в Trainer) — Adam с корректным weight decay.

**Learning rate:** `1e-5` с linear warmup (200 шагов), затем линейное затухание до 0.

**Регуляризация:**
- `weight_decay=0.01` — L2-регуляризация
- `label_smoothing=0.1` — сглаживание меток, снижает переобучение

**Обучение:** 10 эпох, батч 32, `fp16` при наличии GPU.

**Выбор лучшей модели:** оценка каждую эпоху по F1, в конце загружается лучший чекпоинт (`load_best_model_at_end`).

**Data collator:** динамический паддинг до длины максимальной последовательности в батче.

In [19]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./ner_model",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    warmup_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Trainer готов")

Trainer готов


In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,1.059376,0.901891,0.565417,0.555556,0.575636
2,0.785813,0.761697,0.893673,0.871501,0.917001
3,0.745457,0.747280,0.937255,0.915709,0.959839
4,0.740935,0.743937,0.954967,0.944954,0.965194
5,0.746442,0.742710,0.963696,0.950521,0.977242
6,0.734246,0.742179,0.960422,0.946684,0.974565
7,0.745901,0.742193,0.965017,0.951823,0.978581
8,0.729505,0.741439,0.963648,0.951697,0.975904
9,0.742877,0.742037,0.963061,0.949285,0.977242
10,0.729466,0.742166,0.963061,0.949285,0.977242


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2340, training_loss=0.8555383413265913, metrics={'train_runtime': 1238.4902, 'train_samples_per_second': 60.218, 'train_steps_per_second': 1.889, 'total_flos': 4874476764072960.0, 'train_loss': 0.8555383413265913, 'epoch': 10.0})

In [28]:
trainer.save_model("./ner_model_final")
tokenizer.save_pretrained("./ner_model_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ner_model_final/tokenizer_config.json', './ner_model_final/tokenizer.json')

## Шаг 7 - Формирование submission для Kaggle

In [23]:
test_df = pd.read_csv("/content/data/private_test_dataset.csv")
print(test_df.shape)
test_df.head(5)

(3552, 2)


,id,text
0,0,Банкомат не запрашивает CVV-код. Для снятия на...
1,1,"Уточните, пожалуйста, ФИО получателя и точную ..."
2,2,Проверьте папку «Спам» в вашей электронной поч...
3,3,"Я пытаюсь обновить данные в личном кабинете, н..."
4,4,Хочу подключить СМС-информирование. Нужно ли у...


In [26]:
model.eval()
model.to(device)

def predict_spans(text: str) -> list:
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        return_offsets_mapping=True,
        return_tensors="pt",
    )

    offset_mapping = encoding.pop("offset_mapping")[0].tolist()

    with torch.no_grad():
        outputs = model(**{k: v.to(device) for k, v in encoding.items()})

    pred_ids = outputs.logits[0].argmax(-1).tolist()

    spans, current_span = [], None

    for pred_id, (tok_start, tok_end) in zip(pred_ids, offset_mapping):
        if tok_start == 0 and tok_end == 0:
            if current_span:
                spans.append(current_span)
                current_span = None
            continue

        label = id2label[pred_id]

        if label == "O":
            if current_span:
                spans.append(current_span)
                current_span = None

        elif label.startswith("B-"):
            if current_span:
                spans.append(current_span)
            current_span = {"start": tok_start, "end": tok_end, "label": label[2:]}

        elif label.startswith("I-"):
            cat = label[2:]
            if current_span and current_span["label"] == cat:
                current_span["end"] = tok_end
            else:
                if current_span:
                    spans.append(current_span)
                current_span = {"start": tok_start, "end": tok_end, "label": cat}

    if current_span:
        spans.append(current_span)

    # Возвращаем оригинальные названия категорий
    return [(s["start"], s["end"], norm2cat.get(s["label"], s["label"])) for s in spans]

In [27]:
results = []
for _, row in test_df.iterrows():
    spans = predict_spans(row["text"])
    prediction = str(spans) if spans else "[]"
    results.append({"id": row["id"], "Prediction": prediction})

submission = pd.DataFrame(results)
submission.to_csv("submission.csv", index=False)
print(f"Готово: {len(submission)} строк")
submission.head(10)

Готово: 3552 строк


,id,Prediction
0,0,[]
1,1,"[(106, 124, 'Номер телефона')]"
2,2,[]
3,3,"[(88, 93, 'Паспортные данные'), (102, 108, 'Па..."
4,4,"[(82, 94, 'Водительское удостоверение')]"
5,5,"[(51, 63, 'Временное удостоверение личности')]"
6,6,[]
7,7,"[(10, 24, 'СНИЛС клиента')]"
8,8,"[(57, 61, 'ПИН код')]"
9,9,"[(101, 155, 'Содержимое магнитной полосы')]"


## Шаг 8 — Загрузка весов модели на HuggingFace Hub

In [29]:
!pip install huggingface_hub -q
from huggingface_hub import notebook_login
notebook_login()

In [30]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path="./ner_model_final",
    repo_id="dancessa/ner-alpha-bank",
    commit_message="Upload fine-tuned PII NER model",
)
print("✅ Готово!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...l_final/model.safetensors:   0%|          |  566kB /  709MB            

  ...l_final/training_args.bin:   1%|1         |  73.0B / 5.14kB            

✅ Готово!
